# California Home Price Modeling with XGBoost

This notebook trains and evaluates two XGBoost regressors on a chronological 2025 split. The deployed list-unaware model predicts sale price without `ListPrice` or `OriginalListPrice`, allowing estimates for properties that are not actively listed.

The list-aware model serves as a benchmark for the predictive advantage of listing-price fields. Raw MLS records are not included in the repository.

## Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# CPU is the portable default. Set XGB_DEVICE=cuda to accelerate a local verification run.
XGB_DEVICE = os.environ.get('XGB_DEVICE', 'cpu')
print(f'XGBoost device: {XGB_DEVICE}')

XGBoost device: cpu


## Load monthly data

January through August 2025 form the training period. September and October 2025 form a later-time test set.

In [2]:
# January through August training months
df1 = pd.read_csv('CRMLSSold202503_filled.csv')    # March
df2 = pd.read_csv('CRMLSSold202504_filled.csv')    # April
df3 = pd.read_csv('CRMLSSold202505_filled.csv')    # May
df4 = pd.read_csv('CRMLSSold202506_filled.csv')    # June
df5 = pd.read_csv('CRMLSSold202507_filled.csv')    # July
df6 = pd.read_csv('CRMLSSold202508_filled-2.csv')  # August
df7 = pd.read_csv('CRMLSSold202501_filled.csv')    # January
df8 = pd.read_csv('CRMLSSold202502_filled.csv')    # February

# September and October test months
tst = pd.read_csv('CRMLSSold202509.csv')
tst2 = pd.read_csv('CRMLSSold202510.csv')

C:\Users\mmiov\AppData\Local\Temp\ipykernel_88776\1176179771.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df4 = pd.read_csv('CRMLSSold202506_filled.csv')    # June


# Merge

In [3]:
# merge train
trn = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8], ignore_index = True)

# merge test
tst = pd.concat([tst, tst2], ignore_index = True)

# Predefined Filters

In [4]:
# pre-filter count
n1 = len(trn)
n2 = len(tst)
print(f'The size of the training data before filtering is: ', n1)
print(f'The size of the testing data before filtering is: ', n2)

The size of the training data before filtering is:  174802
The size of the testing data before filtering is:  45676


In [5]:
# applying filters to train and test sets
trn = trn[(trn['PropertyType'] == 'Residential') & (trn['PropertySubType'] == 'SingleFamilyResidence') & (trn['StateOrProvince'] == 'CA')]
tst = tst[(tst['PropertyType'] == 'Residential') & (tst['PropertySubType'] == 'SingleFamilyResidence') & (tst['StateOrProvince'] == 'CA')]

In [6]:
# count of dropped instances
print(f'Filtering dropped' , n1 - len(trn), 'instances from the training data')
print(f'Filtering dropped' , n2 - len(tst), 'instances from the testing data')

# new sizes of train and test
print(f'The size of the training data after filtering is: ', len(trn))
print(f'The size of the testing data after filtering is: ', len(tst))

Filtering dropped 88279 instances from the training data
Filtering dropped 22192 instances from the testing data
The size of the training data after filtering is:  86523
The size of the testing data after filtering is:  23484


# Processing

## Retain Only Relevant Features

The deployed feature set contains:

- `DaysOnMarket`

- `Latitude`

- `Longitude`

- `BathroomsTotalInteger`

- `LivingArea`

- `FireplaceYN`

- `YearBuilt`

- `ParkingTotal`

- `BedroomsTotal`

- `PoolPrivateYN`

- `LotSizeAcres`

- `Stories`

Several additional columns remain temporarily to support preprocessing and are removed before training.


## Random Sampling Imputation

In [7]:
retained = [# --- Keep ---
            'AttachedGarageYN',
            'PoolPrivateYN',
            'NewConstructionYN',
            'GarageSpaces',
            'LotSizeSquareFeet',
            'LotSizeAcres',
            'LotSizeArea',
            'YearBuilt',
            'FireplaceYN',
            'LivingArea',
            'BathroomsTotalInteger',
            'Latitude',
            'Longitude',
            'ParkingTotal',
            'CloseDate',
            'ClosePrice',
            'ListingKey',
            'CountyOrParish',
            'DaysOnMarket',
            'ListPrice',
            'BedroomsTotal',
            'ContractStatusChangeDate',
            'City',
            'OriginalListPrice',
            'Stories']

# keeping only retained cols in tst and trn
tst = tst[retained].copy()
trn = trn[retained].copy()

In [8]:
# random sampling impute function
def random_sample_impute(train_col: pd.Series, test_col: pd.Series) -> tuple[pd.Series, pd.Series]:

    nonnull = train_col.dropna()                                # get observed (non-missing) training values
    out_trn = train_col.copy()                                  # copy training column
    out_tst = test_col.copy()                                   # copy testing column

    n_missing_trn = out_trn.isna().sum()                        # count missing in train
    n_missing_tst = out_tst.isna().sum()                        # count missing in test

    sampled_trn = nonnull.sample(n_missing_trn, replace=True).values   # sample from observed train values
    sampled_tst = nonnull.sample(n_missing_tst, replace=True).values   # sample from observed train values

    out_trn.loc[out_trn.isna()] = sampled_trn                   # fill missing in train
    out_tst.loc[out_tst.isna()] = sampled_tst                   # fill missing in test

    return out_trn, out_tst

In [9]:
# reproducibility
np.random.seed(420)

# nan cols
nan_cols = ['AttachedGarageYN',
            'NewConstructionYN',
            'PoolPrivateYN',
            'GarageSpaces',
            'LotSizeSquareFeet',
            'LotSizeAcres',
            'LotSizeArea',
            'OriginalListPrice',
            'YearBuilt',
            'City',
            'FireplaceYN',
            'LivingArea',
            'BathroomsTotalInteger',
            'Longitude',
            'Latitude',
            'ParkingTotal',
            'Stories']

# random sampling impute loop
for col in nan_cols:
    trn[col], tst[col] = random_sample_impute(trn[col], tst[col])

In [10]:
# percentage missingness
print((trn.isna().mean() * 100).sort_values(ascending=False).head(10))
print((tst.isna().mean() * 100).sort_values(ascending=False).head(10))

AttachedGarageYN     0.0
PoolPrivateYN        0.0
NewConstructionYN    0.0
GarageSpaces         0.0
LotSizeSquareFeet    0.0
LotSizeAcres         0.0
LotSizeArea          0.0
YearBuilt            0.0
FireplaceYN          0.0
LivingArea           0.0
dtype: float64
AttachedGarageYN     0.0
PoolPrivateYN        0.0
NewConstructionYN    0.0
GarageSpaces         0.0
LotSizeSquareFeet    0.0
LotSizeAcres         0.0
LotSizeArea          0.0
YearBuilt            0.0
FireplaceYN          0.0
LivingArea           0.0
dtype: float64


## Incorrect Values

### Duplicates

In [11]:
# check duplicate count
print(f"Duplicate count (trn): {trn['ListingKey'].duplicated().sum()}")

# drop duplicates based on most recent close date
trn["CloseDate"] = pd.to_datetime(trn["CloseDate"])  # ensure datetime format
trn = (trn
       .sort_values("CloseDate", ascending=False)    # newest first
       .drop_duplicates(subset="ListingKey", keep="first"))

# verify removal
print(f"Duplicate count after drop (trn): {trn['ListingKey'].duplicated().sum()}")

Duplicate count (trn): 57
Duplicate count after drop (trn): 0


In [12]:
# check duplicate count
print(f"Duplicate count (tst): {tst['ListingKey'].duplicated().sum()}")

# drop duplicates based on most recent close date
tst["CloseDate"] = pd.to_datetime(tst["CloseDate"])
tst = (tst
       .sort_values("CloseDate", ascending=False)
       .drop_duplicates(subset="ListingKey", keep="first"))

# verify removal
print(f"Duplicate count after drop (tst): {tst['ListingKey'].duplicated().sum()}")

Duplicate count (tst): 5
Duplicate count after drop (tst): 0


In [13]:
# drop listingkey and closedate from trn and tst
trn.drop('ListingKey', axis = 1, inplace = True)
tst.drop('ListingKey', axis = 1, inplace = True)
trn.drop('CloseDate', axis = 1, inplace = True)
tst.drop('CloseDate', axis = 1, inplace = True)

### Impossible Values

In [14]:
# impossible value filters

# remove impossible or illogical property values
trn = trn[(trn['BedroomsTotal'] > 0) &
          (trn['BathroomsTotalInteger'] > 0) &
          (trn['LivingArea'] > 0) &
          (trn['LotSizeAcres'] > 0) &
          (trn['LotSizeSquareFeet'] > 0) &
          (trn['ParkingTotal'] >= 0) &
          (trn['YearBuilt'] <= pd.Timestamp.now().year) &
          (trn['YearBuilt'] >= 1800)]

tst = tst[(tst['BedroomsTotal'] > 0) &
          (tst['BathroomsTotalInteger'] > 0) &
          (tst['LivingArea'] > 0) &
          (tst['LotSizeAcres'] > 0) &
          (tst['LotSizeSquareFeet'] > 0) &
          (tst['ParkingTotal'] >= 0) &
          (tst['YearBuilt'] <= pd.Timestamp.now().year) &
          (tst['YearBuilt'] >= 1800)]

# remove negative days on market
trn = trn[trn['DaysOnMarket'] >= 0]

tst = tst[tst['DaysOnMarket'] >= 0]

# keep only properties within California lat/long bounds
trn = trn[(trn['Latitude'].between(32, 42)) &
          (trn['Longitude'].between(-124, -114))]

tst = tst[(tst['Latitude'].between(32, 42)) &
          (tst['Longitude'].between(-124, -114))]

# remove zero or impossible prices
trn = trn[(trn['ClosePrice'] > 0) &
          (trn['ListPrice'] > 0) &
          (trn['OriginalListPrice'] > 0)]

tst = tst[(tst['ClosePrice'] > 0) &
          (tst['ListPrice'] > 0) &
          (tst['OriginalListPrice'] > 0)]

# living area cannot exceed lot size
trn = trn[trn['LivingArea'] <= trn['LotSizeSquareFeet']]

tst = tst[tst['LivingArea'] <= tst['LotSizeSquareFeet']]

# garage spaces cannot exceed total parking
trn = trn[trn['GarageSpaces'] <= trn['ParkingTotal']]

tst = tst[tst['GarageSpaces'] <= tst['ParkingTotal']]

# garagespace and parking totall cannot exceed 30
trn = trn[(trn['GarageSpaces'] <= 30) &
          (trn['ParkingTotal'] <= 30)]

tst = tst[(tst['GarageSpaces'] <= 30) &
          (tst['ParkingTotal'] <= 30)]

### Outlier treatment

Selected numeric fields are trimmed using the 0.5th and 99.5th percentile bounds learned from training data only. The same fixed bounds are applied to the test months.

The final model does not use the earlier IQR experiment. Percentile trimming preserves the middle 99% of each selected training feature and avoids learning thresholds from test data.

In [15]:
num_cols = ['LotSizeSquareFeet',
            'LotSizeAcres',
            'LivingArea',
            'BathroomsTotalInteger',
            'BedroomsTotal',
            'ListPrice',
            'OriginalListPrice',
            'ClosePrice',
            'DaysOnMarket']

pct_bounds = {}
for col in num_cols:
    # Learn the 0.5th and 99.5th percentile thresholds from training data only.
    lower = trn[col].quantile(0.005)
    upper = trn[col].quantile(0.995)
    pct_bounds[col] = (lower, upper)
    trn = trn[(trn[col] >= lower) & (trn[col] <= upper)]

# Apply the fixed training thresholds to test data.
for col, (lower, upper) in pct_bounds.items():
    tst = tst[(tst[col] >= lower) & (tst[col] <= upper)]

## Retain the modeling features

Two variants are trained. The list-aware comparison includes listing-price fields, which closely track final sale price. The deployed list-unaware model removes both listing-price fields and uses the 12 property features exposed by the application.

In [16]:
retained = ['DaysOnMarket',
            'Latitude',
            'Longitude',
            'BathroomsTotalInteger',
            'LivingArea',
            'FireplaceYN',
            'YearBuilt',
            'ParkingTotal',
            'BedroomsTotal',
            'PoolPrivateYN',
            'LotSizeAcres',
            'Stories',
            'ClosePrice',
            'OriginalListPrice',
            'ListPrice']

# keeping only retained cols in tst and trn
tst = tst[retained].copy()
trn = trn[retained].copy()

## Reproducibility boundary

Processed train and test records are not exported from this notebook. The source MLS data and listing-level derivatives remain outside the repository.

## Target and feature encoding

In [17]:
# log ClosePrice
trn['LogPrice'] = np.log(trn['ClosePrice'])
tst['LogPrice'] = np.log(tst['ClosePrice'])

# boolean encoding
bool_cols = ['PoolPrivateYN', 'FireplaceYN']
trn[bool_cols] = trn[bool_cols].astype(int)
tst[bool_cols] = tst[bool_cols].astype(int)

# feature matrices
X_trn_list = trn.drop(columns=['ClosePrice', 'LogPrice'])
X_tst_list = tst.drop(columns=['ClosePrice', 'LogPrice'])

y_trn_list = trn['ClosePrice']
y_tst_list = tst['ClosePrice']

y_trn_log_list = trn['LogPrice']
y_tst_log_list = tst['LogPrice']

## Benchmark: list-aware XGBoost

In [18]:
# Benchmark model: listing-price features are included.
xgb_list = XGBRegressor(max_depth=7,
                        learning_rate=0.01,
                        n_estimators=1000,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        random_state=42,
                        n_jobs=-1,
                        device=XGB_DEVICE)

xgb_list.fit(X_trn_list, y_trn_log_list)
y_pred_log_list = xgb_list.predict(X_tst_list)
y_pred_list = np.exp(y_pred_log_list)

r2_xgb_list = r2_score(y_tst_list, y_pred_list)
mape_xgb_list = mean_absolute_percentage_error(y_tst_list, y_pred_list) * 100
mdape_xgb_list = np.median(np.abs((y_tst_list - y_pred_list) / y_tst_list)) * 100

In [19]:
print(f"R2: {r2_xgb_list:.2f}")
print(f"MAPE: {mape_xgb_list:.2f}%")
print(f"Median Absolute Percentage Error: {mdape_xgb_list:.2f}%")

R2: 0.99
MAPE: 3.30%
Median Absolute Percentage Error: 2.10%


## Hyperparameter selection

The deployed settings came from a two-stage exhaustive five-fold search on a predecessor list-unaware feature frame and were retained after the runtime contract was reduced to the 12 inputs below. `xgboost_tuning.ipynb` contains the grids, archived selections, and reproducibility boundary.

## Deployed model: list-unaware XGBoost

In [20]:
# features (no ListPrice or OriginalListPrice)
X_trn_nolist = trn.drop(columns=['ClosePrice',
                                 'LogPrice',
                                 'ListPrice',
                                 'OriginalListPrice'])

X_tst_nolist = tst.drop(columns=['ClosePrice',
                                 'LogPrice',
                                 'ListPrice',
                                 'OriginalListPrice'])

# targets
y_trn_nolist = trn['ClosePrice']
y_tst_nolist = tst['ClosePrice']

y_trn_log_nolist = trn['LogPrice']
y_tst_log_nolist = tst['LogPrice']

In [21]:
# Primary deployed model: no listing-price features.
xgb_nolist = XGBRegressor(max_depth=7,
                          learning_rate=0.05,
                          n_estimators=1300,
                          subsample=0.8,
                          colsample_bytree=0.8,
                          random_state=42,
                          n_jobs=-1,
                          device=XGB_DEVICE)

xgb_nolist.fit(X_trn_nolist, y_trn_log_nolist)
y_pred_log_nolist = xgb_nolist.predict(X_tst_nolist)
y_pred_nolist = np.exp(y_pred_log_nolist)

r2_xgb_nolist = r2_score(y_tst_nolist, y_pred_nolist)
mape_xgb_nolist = mean_absolute_percentage_error(y_tst_nolist, y_pred_nolist) * 100
mdape_xgb_nolist = np.median(np.abs((y_tst_nolist - y_pred_nolist) / y_tst_nolist)) * 100

In [22]:
print(f"R2: {r2_xgb_nolist:.2f}")
print(f"MAPE: {mape_xgb_nolist:.2f}%")
print(f"Median Absolute Percentage Error: {mdape_xgb_nolist:.2f}%")

R2: 0.90
MAPE: 11.19%
Median Absolute Percentage Error: 7.75%


## Performance Metric Table

In [23]:
results_df = pd.DataFrame({
    'Model': ['List-Aware XGBoost', 'List-Unaware XGBoost'],
    'R2': [round(r2_xgb_list, 2), round(r2_xgb_nolist, 2)],
    'MAPE (%)': [round(mape_xgb_list, 2), round(mape_xgb_nolist, 2)],
    'MdAPE (%)': [round(mdape_xgb_list, 2), round(mdape_xgb_nolist, 2)]
})

print()
print('FINAL MODEL COMPARISON')
print()
print(results_df.to_string(index=False))


FINAL MODEL COMPARISON

               Model   R2  MAPE (%)  MdAPE (%)
  List-Aware XGBoost 0.99      3.30       2.10
List-Unaware XGBoost 0.90     11.19       7.75


## Feature Importance

In [24]:
# list-aware feature importance
feat_importance_list = pd.DataFrame({
    "Feature": X_trn_list.columns,
    "Importance_List (%)": (xgb_list.feature_importances_ * 100).round(2)
}).sort_values("Importance_List (%)", ascending=False)

print("\nFEATURE IMPORTANCE: XGB (With ListPrice)\n")
print(feat_importance_list.to_string(index=False))


FEATURE IMPORTANCE: XGB (With ListPrice)

              Feature  Importance_List (%)
            ListPrice            71.860001
    OriginalListPrice            23.360001
BathroomsTotalInteger             1.010000
           LivingArea             0.870000
             Latitude             0.820000
            Longitude             0.740000
        PoolPrivateYN             0.630000
         DaysOnMarket             0.220000
          FireplaceYN             0.150000
            YearBuilt             0.130000
         ParkingTotal             0.080000
        BedroomsTotal             0.050000
         LotSizeAcres             0.050000
              Stories             0.040000


In [25]:
# list-unaware feature importance
feat_importance_nolist = pd.DataFrame({
    "Feature": X_trn_nolist.columns,
    "Importance_NoList (%)": (xgb_nolist.feature_importances_ * 100).round(2)
}).sort_values("Importance_NoList (%)", ascending=False)

print("\nFEATURE IMPORTANCE: XGB (No ListPrice)\n")
print(feat_importance_nolist.to_string(index=False))


FEATURE IMPORTANCE: XGB (No ListPrice)

              Feature  Importance_NoList (%)
BathroomsTotalInteger                  31.02
             Latitude                  16.66
            Longitude                  14.45
           LivingArea                  13.22
          FireplaceYN                   7.37
        PoolPrivateYN                   7.04
            YearBuilt                   2.98
         ParkingTotal                   1.85
        BedroomsTotal                   1.60
         LotSizeAcres                   1.57
              Stories                   1.31
         DaysOnMarket                   0.94


In [26]:
X_trn_nolist.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78162 entries, 130216 to 145797
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DaysOnMarket           78162 non-null  int64  
 1   Latitude               78162 non-null  float64
 2   Longitude              78162 non-null  float64
 3   BathroomsTotalInteger  78162 non-null  float64
 4   LivingArea             78162 non-null  float64
 5   FireplaceYN            78162 non-null  int64  
 6   YearBuilt              78162 non-null  float64
 7   ParkingTotal           78162 non-null  float64
 8   BedroomsTotal          78162 non-null  float64
 9   PoolPrivateYN          78162 non-null  int64  
 10  LotSizeAcres           78162 non-null  float64
 11  Stories                78162 non-null  float64
dtypes: float64(9), int64(3)
memory usage: 7.8 MB


## Price Bands

In [27]:
bands = {
    'Below $500K': (0, 500_000),
    '$500K to $1M': (500_000, 1_000_000),
    '$1M to $2M': (1_000_000, 2_000_000),
    '$2M to $5M': (2_000_000, 5_000_000),
    '$5M to $10M': (5_000_000, 10_000_000),
    '$10M and above': (10_000_000, np.inf),
}

def mdape_fn(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    return round(np.median(np.abs((y_true - y_pred) / y_true) * 100), 2)

def mape_fn(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    return round(np.mean(np.abs((y_true - y_pred) / y_true) * 100), 2)

rows = []
for band_name, (low, high) in bands.items():
    mask_list = (y_tst_list >= low) & (y_tst_list < high)
    mask_nolist = (y_tst_nolist >= low) & (y_tst_nolist < high)
    rows.append([
        band_name,
        mdape_fn(y_tst_nolist[mask_nolist], y_pred_nolist[mask_nolist]),
        mape_fn(y_tst_nolist[mask_nolist], y_pred_nolist[mask_nolist]),
        mdape_fn(y_tst_list[mask_list], y_pred_list[mask_list]),
        mape_fn(y_tst_list[mask_list], y_pred_list[mask_list]),
    ])

rows.append([
    'All price bands',
    mdape_fn(y_tst_nolist, y_pred_nolist),
    mape_fn(y_tst_nolist, y_pred_nolist),
    mdape_fn(y_tst_list, y_pred_list),
    mape_fn(y_tst_list, y_pred_list),
])

price_band_table = pd.DataFrame(
    rows,
    columns=['Price band', 'No-list MdAPE', 'No-list MAPE',
             'List-aware MdAPE', 'List-aware MAPE'],
)

print()
print('PRICE-BAND PERFORMANCE')
print()
print(price_band_table.to_string(index=False))


PRICE-BAND PERFORMANCE

     Price band  No-list MdAPE  No-list MAPE  List-aware MdAPE  List-aware MAPE
    Below $500K           7.40         12.96              2.01             3.45
   $500K to $1M           6.54          9.64              1.74             2.77
     $1M to $2M           9.07         11.85              2.59             3.73
     $2M to $5M          11.52         13.89              2.95             4.17
    $5M to $10M            NaN           NaN               NaN              NaN
 $10M and above            NaN           NaN               NaN              NaN
All price bands           7.75         11.19              2.10             3.30


## Artifact provenance

The deployed model is stored as `artifacts/xgb_nolist.ubj`, XGBoost's native UBJSON format. It was converted from the repository's trusted pickle with an exact parity check across 303 feature rows. The application verifies the native artifact checksum and embedded feature order before inference.

The list-unaware model is deployed by the Streamlit application. Reported performance is tied to the chronological September-October 2025 test period shown above. September appeared in predecessor tuning diagnostics before October became available, so the combined period is not a strictly untouched evaluation set. Application smoke tests verify software behavior, not predictive performance.